In [ ]:
import os
import glob
import math
import json
import gradio as gr
from dotenv import load_dotenv

# LangChain & LLM components
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True)
DB_NAME = "vector_db"
MODEL = "gpt-4.1-nano"

## 1. Document Ingestion & Vector Database Creation
Instead of complex modular files, we wrap the database building process into a single function. It reads Markdown files from `knowledge-base/`, chunks them, and stores them in Chroma.

In [ ]:
def build_database():
    print("Loading documents from 'knowledge-base/'...")
    folders = glob.glob("knowledge-base/*")
    documents = []
    
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
        for doc in loader.load():
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)

    print(f"Loaded {len(documents)} documents. Splitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Refresh DB if it already exists
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    print("Storing in Chroma vector store...")
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
    print("✅ Database built successfully!")
    return vectorstore

# NOTE: Run this once to create the vector_db folder.
# vectorstore = build_database()

## 2. RAG Pipeline (Retrieval + Generation)
We set up our Retriever and LLM, then use a standard prompt template to combine the user's question with the retrieved chunks.

In [ ]:
# Initialize components
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant for Insurellm.
If relevant, use the given context to answer the question.
If you don't know the answer, say so.

Context:
{context}
"""

def answer_question(question: str):
    # 1. Retrieve the most relevant document chunks
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    
    # 2. Insert context into the prompt
    formatted_prompt = SYSTEM_PROMPT.format(context=context)
    
    # 3. Generate answer
    response = llm.invoke([
        SystemMessage(content=formatted_prompt), 
        HumanMessage(content=question)
    ])
    return response.content, docs

# Test the function
# answer, docs = answer_question("Who is Avery?")
# print(answer)

## 3. Gradio Chat Interface
A simple wrapper function to connect our `answer_question` logic to a clean UI.

In [ ]:
def chat_handler(message, history):
    answer, _ = answer_question(message)
    return answer

# Uncomment to launch the web interface locally
# gr.ChatInterface(chat_handler).launch(inbrowser=True)

## 4. Evaluation (LLM-as-a-Judge)
To evaluate the system without overcomplicating things with Pydantic classes, we use simple standard Python dictionaries. This tests retrieval (MRR) and uses the LLM to grade itself.

In [ ]:
def calculate_mrr(keyword: str, retrieved_docs: list) -> float:
    """Calculates Mean Reciprocal Rank to see how highly the keyword was ranked in search results."""
    keyword_lower = keyword.lower()
    for rank, doc in enumerate(retrieved_docs, start=1):
        if keyword_lower in doc.page_content.lower():
            return 1.0 / rank
    return 0.0

def evaluate_pipeline(question: str, keywords: list, reference_answer: str):
    """Runs a full evaluation on a single question using dictionaries for simplicity."""
    
    # --- 1. Evaluate Retrieval ---
    generated_answer, retrieved_docs = answer_question(question)
    mrr_scores = [calculate_mrr(kw, retrieved_docs) for kw in keywords]
    avg_mrr = sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0
    
    # --- 2. Evaluate Generation (LLM Judge) ---
    judge_prompt = f"""Evaluate this generated answer against the reference.
Question: {question}
Generated Answer: {generated_answer}
Reference Answer: {reference_answer}

Score Accuracy, Completeness, and Relevance from 1 to 5. Return ONLY a JSON dictionary exactly like this:
{{"accuracy": 5, "completeness": 4, "relevance": 5, "feedback": "Good answer but missed a detail."}}"""

    try:
        judge_response = llm.invoke([HumanMessage(content=judge_prompt)])
        # Clean the response string just in case the LLM wrapped it in markdown code blocks
        cleaned_json = judge_response.content.strip().strip("```json").strip("```")
        answer_eval = json.loads(cleaned_json)
    except Exception as e:
        answer_eval = {"error": "Failed to parse JSON", "raw": judge_response.content}

    return {
        "retrieval": {
            "mrr": round(avg_mrr, 3),
            "keywords_found": sum(1 for s in mrr_scores if s > 0),
            "total_keywords": len(keywords)
        },
        "generation": answer_eval
    }

# Example Usage:
# result = evaluate_pipeline(
#     question="Who won the prestigious IIOTY award in 2023?",
#     keywords=["Maxine", "Thompson", "IIOTY"],
#     reference_answer="Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023."
# )
# print(json.dumps(result, indent=2))